# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides an example workflow for loading, exploring, and processing data from the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object)
print(f"Name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Review available record sets, their fields, and unique `@id` values. This helps us identify how the data is structured and which parts to extract.

*Each entity (record set, field, column) is referenced by its `@id` as per Croissant best practices.*

In [ ]:
# Get and display the dataset's record sets and fields, referencing them by @id
record_sets = list(dataset.record_sets.values())
if not record_sets:
    print("No record sets are registered in the Croissant schema.\nTry checking the documentation or schema directly.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        print("  Fields:")
        for field in rs.get('fields', []):
            print(f"    {field['@id']} (name: {field.get('name', '<no name>')}, dataType: {field.get('dataType', '<unknown>')})")
        print("  Columns:")
        for col in rs.get('columns', []):
            print(f"    {col['@id']} (name: {col.get('name', '<no name>')})")
        print('-' * 40)

# Demo: Print a sample record (for the first record set found)
if record_sets:
    first_rs_id = record_sets[0]['@id']
    print(f"\nFirst record from record set {first_rs_id}:")
    for rec in dataset.records(record_set=first_rs_id):
        print(rec)
        break
else:
    print("No record sets found in record_sets.")

## 3. Data Extraction

Load data from each relevant record set into a Pandas DataFrame. Use record set and field `@id` values from the previous overview.

We'll collect all record sets with tabular data (excluding auxiliary documentation sets if present).

In [ ]:
# Extract all record set @id values
rs_ids = [rs['@id'] for rs in dataset.record_sets.values()] if dataset.record_sets else []
print('All record set @id values:')
print(rs_ids)

dataframes = {}

for rs_id in rs_ids:
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f'Loaded DataFrame for {rs_id}: shape {df.shape}')
    except Exception as e:
        print(f'Error loading records for {rs_id}: {str(e)}')

# Show columns for the first loaded DataFrame (by record set @id)
if dataframes:
    example_rs_id = list(dataframes.keys())[0]
    print(f"\nColumns in {example_rs_id}:\n{dataframes[example_rs_id].columns.tolist()}")
    dataframes[example_rs_id].head()
else:
    print('No dataframes loaded - check if the Croissant schema defines record sets and if they have data.')

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps:
- **Filtering** on a numeric field 
- **Normalization** of numeric fields
- **Grouping** by a key attribute

Each column is referenced by its `@id` as required.

In [ ]:
# Choose the record set and numeric field for analysis (replace with actual @id as seen in data overview)
if not dataframes:
    print("No dataframes available for EDA.")
else:
    record_set_id = example_rs_id
    df = dataframes[record_set_id]

    # Try to auto-detect a numeric column (prefer one with 'log_likelihood', 'coefficient', or 'age', etc. in its name/@id)
    import numpy as np
    numeric_field_id = None
    for col in df.columns:
        if any(x in col.lower() for x in ['likelihood', 'loglikelihood', 'coefficient', 'score', 'value', 'age']):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Fall back: use first column with numeric dtype
        num_cols = df.select_dtypes(include=[np.number]).columns
        numeric_field_id = num_cols[0] if len(num_cols) else df.columns[0]

    print(f"Numeric field selected for analysis (by @id): {numeric_field_id}")

    # Set threshold for filtering (mean + 1 std if possible)
    if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean() + df[numeric_field_id].std()
        filtered_df = df[df[numeric_field_id] > threshold]
    else:
        threshold = 10 # fallback
        try:
            filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
        except:
            filtered_df = df.iloc[:0]

    print(f"\nFiltered records with {numeric_field_id} > {threshold:.3f}: {len(filtered_df)} records\n")
    display(filtered_df.head())

    # Normalize numeric field in filtered records
    mean_val = filtered_df[numeric_field_id].mean() if not filtered_df.empty else 0
    std_val = filtered_df[numeric_field_id].std() if not filtered_df.empty else 1
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / (std_val if std_val != 0 else 1)
    print(f"\nNormalized {numeric_field_id} for filtered records:\n")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a field (e.g., gender, 'ward', or any categorical field @id)
    group_field = None
    for col in df.columns:
        if any(x in col.lower() for x in ['gender', 'ward', 'group', 'county']):
            group_field = col
            break
    if group_field:
        print(f"\nGrouping filtered data by '{group_field}':\n")
        group_summary = filtered_df.groupby(group_field).mean(numeric_only=True)
        display(group_summary)
    else:
        print("No suitable grouping field detected.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to a selected group field if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No dataframes available for visualization.")
else:
    df = dataframes[record_set_id]

    # Histogram of the selected numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20, color='skyblue')
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If a group_field was found, plot boxplot by group
    if group_field is not None:
        plt.figure(figsize=(9,5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df, palette='Set2')
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
    else:
        print("No group field found for grouped visualization.")

## 6. Conclusion

- Successfully loaded the dataset and explored its structure via Croissant `@id` references.
- Record sets and fields were examined; example columnar data was loaded and analyzed.
- Applied normalization and grouped aggregation for key numeric fields.
- Visualized data distributions and group-level patterns (when suitable fields were present).

Further exploration can leverage the full richness of the Croissant metadata, relationships, and provenance to support reproducible ML and data workflows on FAIR datasets.